<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/1_2-firm-size-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook analyses the various variables associated with firm size.

**Recommended runtime:** CPU

# Setup

In [ ]:
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/digi-inno-road-prod'
    if os.path.isdir(repo_path):
      cwd = os.getcwd()
      os.chdir(repo_path)
      !git pull
      os.chdir(cwd)
    else:
      !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git /content/drive/MyDrive/digi-inno-road-prod
      print('Repository cloned into your Google Drive. It is strongly recommended that you copy the credentials.json, sheet.json, and token.json files into the secrets folder before proceeding.')
    sys.path.insert(0, repo_path)
    IN_COLAB = True
except ImportError:
    repo_path = os.path.abspath(os.path.join('../src'))
    IN_COLAB = False

if not repo_path in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
if IN_COLAB:
  output_path = os.path.join(repo_path, 'analysis', 'outputs')
else:
  output_path = os.path.join('.', 'outputs')
output_path = os.path.abspath(output_path)

# Import packages & data

Conda packages

In [ ]:
import matplotlib
import numpy
import pandas
import sklearn

Project sourcecode packages

In [ ]:
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps, wrangle_grants

Data

In [ ]:
data = get_sheet_dfs()

INCLUDE_NO_GRANTS_FIRMS = True
if INCLUDE_NO_GRANTS_FIRMS:
  roadmaps_df = pandas.concat([data['Roadmaps'], data['RoadmapsWithoutGrants']]).reset_index(drop=True)
else:
  roadmaps_df = data['Roadmaps']
roadmaps_df = wrangle_roadmaps(roadmaps_df)

grants_df = wrangle_grants(data['Grants'])

# Analysis

## Analysis setup

In [ ]:
org_size_col = 'Org Size by Number of FTE (calc)'
num_fte_col = 'Number of FTE Employees (calc)'
num_ft_col = 'Number of FT employees'
num_pt_col = 'Number of PT employees'
turnover_col = 'Turnover'
size_cols = [org_size_col, num_fte_col, num_ft_col, num_pt_col, turnover_col]
# roadmaps_df[size_cols]

## Org Size alone

We have "Org Size" categorical data for this many firms:

In [ ]:
int(roadmaps_df[org_size_col].notna().sum())

They are distributed as such:

In [ ]:
roadmaps_df[roadmaps_df[org_size_col].notna()][org_size_col].value_counts()

## Numbers of FTE, FT, and PT

We have a valid "Number of FTE" for this many firms:

In [ ]:
valid_fte_mask = roadmaps_df[num_fte_col].notna()
int(valid_fte_mask.sum())

However, many of these values are zeros. Excluding the zeros leaves this many firms:

In [ ]:
valid_fte_mask = (roadmaps_df[num_fte_col] > 0.0).fillna(False)
int(valid_fte_mask.sum())

The numbers of FT or PT employees appear to be free of the spurious zeros afflicting the "Number of FTE employees" column.

In [ ]:
int((roadmaps_df[num_ft_col] == 0.0).fillna(False).sum())

In [ ]:
int((roadmaps_df[num_pt_col] == 0.0).fillna(False).sum())

Looking at the missing or zero `Number of FTE` values, we find that this matches up perfectly with missing values in **both** the `Number of FT employees` and the `Number of PT employees` columns

In [ ]:
invalid_fte_mask = (roadmaps_df[num_fte_col] == 0.0).fillna(True)
bool((roadmaps_df[invalid_fte_mask][num_ft_col].isna() & roadmaps_df[invalid_fte_mask][num_pt_col].isna()).all())

Wherever
1. the `Number of FTE` is given, and
2. the `Number of FT employees` is given, but
3. the `Number of PT employees` is missing or zero,

the `Number of FTE` and the `Number of FT employees` match.

In [ ]:
roadmaps_df[valid_fte_mask & (roadmaps_df[num_pt_col] == 0.0).fillna(True)][size_cols]

In [ ]:
mask = valid_fte_mask & (roadmaps_df[num_pt_col] == 0.0).fillna(True) & roadmaps_df[num_ft_col].notna()
bool((roadmaps_df[mask][num_fte_col] == roadmaps_df[mask][num_ft_col]).all())

In the small number of cases where the `Number of PT employees` is given, each PT employee appears to be assumed to be equivalent to half of one FT employee when calculating the `Number of FTE employees`

In [ ]:
mask = (roadmaps_df[num_pt_col] > 0.0).fillna(False)
roadmaps_df[mask][size_cols]

In [ ]:
roadmaps_df[mask][num_fte_col] == (roadmaps_df[mask][num_ft_col].fillna(0.0) + 0.5*roadmaps_df[mask][num_pt_col])

Overall, we conclude that `Number of FTE employees (calc)` is a fair summation of the `Number of FT employees` and the `Number of PT employees` columns.

## Org Size and Number of FTE employees

There are no firms for which the `Number of FTE employees` is given and above zero, but for which the `Org Size` category is missing.

In [ ]:
bool((valid_fte_mask & roadmaps_df[org_size_col].isna()).any())

However, there are this many firms where the `Org Size` category is given but the `Number of FTE employees` is missing or zero:

In [ ]:
int((invalid_fte_mask & roadmaps_df[org_size_col].notna()).sum())

If we look at the firms where both the `Org Size` category is given and the `Number of FTE employees` is given and greater than zero, we find that the categories do not always match up:

In [ ]:
roadmaps_df[mask].groupby(org_size_col, observed=True)[num_fte_col].apply(lambda grp: {'Min': grp.min(), 'Max': grp.max(), 'Count': grp.count()})

This many firms labeled with an `Org Size` of `Small - 10-49` have more than 49:

In [ ]:
len(roadmaps_df[mask & (roadmaps_df[org_size_col] == 'Small - 10-49') & (roadmaps_df[num_fte_col] > 49)])

And this many firms labeled with an `Org Size` of `Medium - 50-249` have more than 249:

In [ ]:
len(roadmaps_df[mask & (roadmaps_df[org_size_col] == 'Medium - 50-249') & (roadmaps_df[num_fte_col] > 249)])

Out of hundreds of firms, so few categorisation errors is negligle, and so we conclude that the `Org Size` category is consistent with the `Number of FTE` columns. `Org Size` lacks precision but has greater coverage.

## Turnover alone

We have Turnover data for many of the firms. Any null values ("£-") were reassigned NaN value at data ingress and formatting.

In [ ]:
roadmaps_df['Turnover'].notna().value_counts()

No firms reported zero turnover.

In [ ]:
bool((roadmaps_df['Turnover'] == 0.0).any())

There is one firm with a reported Turnover of £500 million, which is far greater than any other.

In [ ]:
roadmaps_df['Turnover'].plot.box()

Inspecting this firm more closely:

In [ ]:
roadmaps_df[roadmaps_df['Turnover'] == roadmaps_df['Turnover'].max()][size_cols]

Plotting the distribution of other firms reveals a rough power law:

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.hist(roadmaps_df[roadmaps_df['Turnover'] < roadmaps_df['Turnover'].max()]['Turnover'], bins=100)
ax.set_xlim(left=0.0)
ax.set_xlabel('Turnover (£)')
ax.set_ylabel('Number of firms')
ax

Given the huge range in values, look at the log-Turnover (base 10 for readability) instead.

In [ ]:
roadmaps_df['Log10 Turnover'] = numpy.log10(roadmaps_df['Turnover'])

In [ ]:
fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.hist(roadmaps_df['Log10 Turnover'], bins=50)
ax.set_xlabel('Turnover (£)')
ax.set_ylabel('Number of firms')

ax.set_xticks(numpy.arange(2, 9, 1), labels=['£100', '£1,000', '£10,000', '£100,000', '£1,000,000', '£10,000,000', '£100,000,000'], rotation=15)
ax

## Number of FTE employees and Turnover

Filtering by those firms where `Number of FTE employees` is given and above zero, and where the `Turnover` is given, we find this many:

In [ ]:
model_df = roadmaps_df[
    (roadmaps_df[num_fte_col] > 0.0).fillna(False)
  ][
    [num_fte_col, turnover_col]
  ].dropna()
len(model_df)

We then fit a Linear Regression model to this subset of the data and draw a scatter plot:

In [ ]:
X = model_df[num_fte_col].values.reshape(-1, 1)
y = model_df[turnover_col].values.reshape(-1, 1)
model = sklearn.linear_model.LinearRegression()
model.fit(X, y)

intercept = float(model.intercept_[0])
coef = float(model.coef_[0][0])

fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.plot(
    [X.min(), X.max()],
    [intercept+(X.min()*coef), intercept+(X.max()*coef)],
    color='red'
)
ax.scatter(X, y)
ax.set_xlabel('Number of FTE Employees')
ax.set_ylabel('Turnover (millions £s)')

yticks = numpy.arange(0, 1.1e8, 1e7)
yticklabels = [str(int(i/1e6)) for i in yticks]
ax.set_yticks(yticks, labels=yticklabels)

ax

In [ ]:
print(f"Full model: intercept {intercept}, coefficient {coef}, R^2 {model.score(X,y)}")

## Org Size and Turnover

In [ ]:
plot_data = [roadmaps_df['Log10 Turnover'].dropna().to_numpy()]

for org_s in ['Micro - 1-9', 'Small - 10-49', 'Medium - 50-249']:
  plot_data.append(roadmaps_df[roadmaps_df['Org Size by Number of FTE (calc)'] == org_s]['Log10 Turnover'].dropna().to_numpy())

fig, ax = matplotlib.pyplot.subplots(figsize=(6,6))
ax.boxplot(plot_data)

ax.set_xlabel('Firm size')
ax.set_xticklabels(['All firms', 'Micro - 1-9', 'Small - 10-49', 'Medium - 50-249'])

ax.set_ylabel('Turnover')
ax.set_yticks(numpy.arange(2, 9, 1), labels=['£100', '£1,000', '£10,000', '£100,000', '£1,000,000', '£10,000,000', '£100,000,000'])

ax

# Conclusion

While `Org Size` lacks the precision of either `Number of FTE employees` or of `Turnover`, it is a suitable proxy for both and has far greater coverage. It also does not have the distrorting outliers that appear to afflict the other two variables (although there is a small number of firms which appear to be mis-categorised, with their `Number of FTE employees` falling outside the range of their `Org Size` category).